# Field Monitoring with Vector Data Cubes

This notebook computes a cloud-free NDVI time series for agricultural fields, with
Sentinel-2 data of the Copernicus Data Space Ecosystem (CDSE). The result of the
zonal statistics is a **vector data cube**: one value for each field and each date.
The back-end processes it further and writes it in the usual GIS formats.

In this notebook, we will:

1. Load field boundaries from a FlatGeobuf file with `load_url`.
2. Move the field edges 10 m inward with `vector_reproject` and `vector_buffer`.
3. Compute NDVI from Sentinel-2 L2A, without clouds (scene classification).
4. Compute the mean NDVI of each field and date with `aggregate_spatial`.
5. Find the seasonal peak of each field with `filter_temporal` and `reduce_dimension`.
6. Divide each date by the peak with `merge_cubes`.
7. Select fields with `filter_bbox` and `filter_vector`.
8. Download the fields in GeoJSON, GeoParquet, GeoPackage, FlatGeobuf and Shapefile.

See [Vector Data Cubes](https://sentinel-hub.github.io/titiler-openeo/vector-cube/) for the processes and formats, and
[ADR 0014](https://github.com/sentinel-hub/titiler-openeo/blob/main/docs/adr/0014-vector-cubes-and-formats.md)
for the design.

## Requirements

- A CDSE account, for the login.
- Python packages: `openeo>=0.54`, `geopandas`, `pyarrow` and `matplotlib`. In this
  repository: `uv sync --group notebook`.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
from openeo.rest import OpenEoApiError

## Connect to the back-end

The back-end is titiler-openeo, with the CDSE STAC catalogue and data.

In [ ]:
import os

import openeo

# The deployment with the vector data cubes. Set TITILER_OPENEO_URL to use
# another one, for example a local server: http://localhost:8082
URL = os.environ.get("TITILER_OPENEO_URL", "https://openeo.ds.io")

# Login with your Copernicus Data Space Ecosystem account (device flow):
# open the link that is printed below and enter the code.
connection = openeo.connect(URL).authenticate_oidc_device()

The next cell stops the notebook if the back-end does not have the vector data cube
features yet.

In [ ]:
processes = {process["id"] for process in connection.list_processes()}
formats = connection.list_file_formats()
missing = sorted(
    {"load_url", "vector_buffer", "vector_reproject", "merge_cubes", "filter_vector"}
    - processes
)
if (
    missing
    or "GeoParquet" not in formats["output"]
    or "FlatGeobuf" not in formats["input"]
):
    raise RuntimeError(
        f"{URL} does not have the vector data cube features yet "
        f"(missing processes: {missing})."
    )
print(f"✅ {URL} supports vector data cubes.")
print(
    "Vector input formats:",
    sorted(k for k, v in formats["input"].items() if "vector" in v["gis_data_types"]),
)

In [ ]:
# The demo fields, as GeoJSON, FlatGeobuf and GeoParquet files. The back-end
# downloads them, so the URL must be public. For a local server, serve the
# folder docs/src/notebooks/data and set DEMO_DATA_URL=http://localhost:8000
DEMO_DATA_URL = os.environ.get(
    "DEMO_DATA_URL",
    "https://raw.githubusercontent.com/sentinel-hub/titiler-openeo/main/docs/src/notebooks/data",
)

# The downloaded results go to this folder.
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)

# The area of the fields, near Mol (Belgium).
BBOX = {"west": 5.05, "south": 51.21, "east": 5.09, "north": 51.24}

# The growing season of the fields.
SEASON = ["2024-03-01", "2024-11-01"]


def callback(process_id, **arguments):
    """Return a callback with one process.

    The openEO Python client has no `reduce_dimension` or `merge_cubes` method on a
    vector data cube, so these calls give the callback as a process graph.
    """
    return {
        "process_graph": {
            process_id: {
                "process_id": process_id,
                "arguments": arguments,
                "result": True,
            }
        }
    }

## 1. Load the fields with `load_url`

`load_url` reads a vector file from a public URL. It accepts GeoJSON, GeoParquet,
GeoPackage (`GPKG`), FlatGeobuf and Shapefile. Here the four fields are a FlatGeobuf
file: the back-end reads it with HTTP range requests. The fields come from the
[openEO Python client tutorial](https://open-eo.github.io/openeo-python-client/).

The result is a vector data cube with a `geometry` dimension and no values. We download
it as GeoJSON to look at it.

In [ ]:
fields = connection.load_url(f"{DEMO_DATA_URL}/mol_fields.fgb", format="FlatGeobuf")

fields.download(OUT / "fields.geojson", format="GeoJSON")
fields_gdf = gpd.read_file(OUT / "fields.geojson")
# FlatGeobuf orders the features by its spatial index: sort them by field.
fields_gdf.sort_values("field_id")

## 2. Move the field edges inward

The pixels at the edge of a field also contain the road or the next field. We move the
edges 10 m inward with `vector_buffer` and a negative distance.

The distance of `vector_buffer` is in metres, so the geometries must be in a CRS in
metres. The fields are in EPSG:4326 (degrees). We reproject them first to UTM zone 31N
(EPSG:32631) with `vector_reproject`.

In [ ]:
fields_utm = fields.process("vector_reproject", data=fields, projection=32631)
inner = fields_utm.process("vector_buffer", geometries=fields_utm, distance=-10)

inner.download(OUT / "fields_inner.gpkg", format="GPKG")
inner_gdf = gpd.read_file(OUT / "fields_inner.gpkg")
print("CRS of the GeoPackage:", inner_gdf.crs)

outline = fields_gdf.to_crs(32631)
fig, (ax_all, ax_zoom) = plt.subplots(1, 2, figsize=(14, 5))
for ax in (ax_all, ax_zoom):
    outline.plot(ax=ax, color="lightgrey", edgecolor="black")
    inner_gdf.plot(ax=ax, color="tab:green", alpha=0.7)
# Zoom on a corner of field A: the grey strip is 10 m wide.
x, y = outline.loc[outline["name"] == "field A"].geometry.iloc[0].exterior.coords[0]
ax_zoom.set_xlim(x - 60, x + 60)
ax_zoom.set_ylim(y - 60, y + 60)
ax_all.set_title("Fields (grey) and 10 m inside the edges (green)")
ax_zoom.set_title("Zoom on a corner of field A")
plt.show()

The GIS formats keep the CRS of the vector data cube: the GeoPackage is in EPSG:32631.

Without `vector_reproject`, the back-end refuses the buffer, and the message names the
UTM zone to use:

In [ ]:
try:
    fields.process("vector_buffer", geometries=fields, distance=-10).download(
        OUT / "error.geojson", format="GeoJSON"
    )
except OpenEoApiError as error:
    print(f"{error.code}: {error.message}")

## 3. NDVI without clouds

We load Sentinel-2 L2A for the season: the red (`B04_10m`) and near-infrared
(`B08_10m`) bands, and the scene classification (`SCL_20m`). The pixels of the SCL
classes 3 (cloud shadow), 8 and 9 (clouds) and 10 (thin cirrus) are masked.

By default, titiler-openeo loads a grid of 1024 pixels over the extent. For this small
area, that is about 3 m: finer than the 10 m of the bands. See
[Concepts](https://sentinel-hub.github.io/titiler-openeo/concepts/) to change the resolution.

In [ ]:
s2 = connection.load_collection(
    "sentinel-2-l2a",
    spatial_extent=BBOX,
    temporal_extent=SEASON,
    bands=["B04_10m", "B08_10m", "SCL_20m"],
    max_cloud_cover=30,
)
scl = s2.band("SCL_20m")
clouds = (scl == 3) | (scl == 8) | (scl == 9) | (scl == 10)
red = s2.band("B04_10m")
nir = s2.band("B08_10m")
ndvi = ((nir - red) / (nir + red)).mask(clouds)

## 4. Zonal statistics: a vector data cube

`aggregate_spatial` computes the mean NDVI of each field for each date. The geometries
are the vector data cube of step 2. The result is a vector data cube with the
dimensions `geometry` and `t`, in EPSG:4326.

We download it as GeoParquet: one row for each field and date, with the properties of
the field (`field_id`, `name`), the date `t` and the `value`. A date without a value
had clouds over the whole field.

This request reads the Sentinel-2 data, so it takes about one minute. If it fails with
a rate limit (HTTP 429) of the STAC catalogue, run the cell again.

In [ ]:
series = ndvi.aggregate_spatial(geometries=inner, reducer="mean")
series.download(OUT / "ndvi_series.parquet", format="GeoParquet")

series_gdf = gpd.read_parquet(OUT / "ndvi_series.parquet")
print(f"CRS: EPSG:{series_gdf.crs.to_epsg()} | rows: {len(series_gdf)}")
table = series_gdf.pivot(index="t", columns="name", values="value")
table.plot(
    marker="o",
    figsize=(10, 4),
    ylabel="Mean NDVI",
    xlabel="Date",
    title="NDVI of each field",
)
plt.show()

## 5. Process the vector data cube: the seasonal peak

Many openEO processes work on the dimensions of a vector data cube other than
`geometry`. Here, `filter_temporal` keeps May to September, and `reduce_dimension`
with the `max` reducer removes the `t` dimension: one value for each field.

We download the result as a GeoPackage and make a map.

In [ ]:
season = series.process(
    "filter_temporal", data=series, extent=["2024-05-01", "2024-10-01"]
)
peak = season.process(
    "reduce_dimension",
    data=season,
    dimension="t",
    reducer=callback("max", data={"from_parameter": "data"}),
)
peak.download(OUT / "ndvi_peak.gpkg", format="GPKG")

peak_gdf = gpd.read_file(OUT / "ndvi_peak.gpkg")
ax = peak_gdf.plot(
    column="value", cmap="YlGn", legend=True, edgecolor="black", figsize=(8, 5)
)
for _, row in peak_gdf.iterrows():
    point = row.geometry.representative_point()
    ax.annotate(f"{row['name']}\n{row['value']:.2f}", (point.x, point.y), ha="center")
ax.set_title("Peak NDVI, May to September 2024")
plt.show()

## 6. `merge_cubes`: NDVI relative to the peak

`merge_cubes` can merge two vector data cubes. The time series has the dimensions
`geometry` and `t`; the peak has only `geometry`. The peak is repeated for each date
(example 4 of the openEO specification), so each cell is in both cubes. The
`overlap_resolver` combines them: here `divide` gives the NDVI as a fraction of the
peak.

Both cubes come from the same Sentinel-2 request, so the back-end reads the data once.

In [ ]:
relative = season.process(
    "merge_cubes",
    cube1=season,
    cube2=peak,
    overlap_resolver=callback(
        "divide", x={"from_parameter": "x"}, y={"from_parameter": "y"}
    ),
)
relative.download(OUT / "ndvi_relative.parquet", format="GeoParquet")

relative_gdf = gpd.read_parquet(OUT / "ndvi_relative.parquet")
relative_table = relative_gdf.pivot(index="t", columns="name", values="value")
print("Maximum of each field (1.0 = the peak date):")
print(relative_table.max().round(3).to_string())
relative_table.plot(marker="o", figsize=(10, 4), ylabel="NDVI / peak", xlabel="Date")
plt.show()

## 7. Select fields

These processes work on the fields only, without Sentinel-2 data:

- `filter_bbox` keeps the geometries that are **fully within** a bounding box.
- `filter_vector` keeps the geometries that have a spatial relation with other
  geometries, for example `intersects`. Here the other geometry comes from
  `load_geojson`.

In [ ]:
east = fields.filter_bbox(west=5.075, south=51.21, east=5.09, north=51.24)
east.download(OUT / "fields_east.geojson", format="GeoJSON")
print(
    "filter_bbox (east part):",
    sorted(gpd.read_file(OUT / "fields_east.geojson")["name"]),
)

area = connection.load_geojson(
    {
        "type": "Polygon",
        "coordinates": [
            [
                [5.05, 51.21],
                [5.066, 51.21],
                [5.066, 51.232],
                [5.05, 51.232],
                [5.05, 51.21],
            ]
        ],
    }
)
west = fields.filter_vector(geometries=area, relation="intersects")
west.download(OUT / "fields_west.geojson", format="GeoJSON")
print(
    "filter_vector (intersects the west area):",
    sorted(gpd.read_file(OUT / "fields_west.geojson")["name"]),
)

## 8. Output formats

The same vector data cube in each GIS format. GeoJSON is always in EPSG:4326 (RFC 7946);
the other formats keep the CRS of the cube (here EPSG:32631). The Shapefile is a ZIP
file. CSV is also available, without the geometries.

In [ ]:
outputs = {
    "GeoJSON": "fields_inner.geojson",
    "GeoParquet": "fields_inner.parquet",
    "GPKG": "fields_inner.gpkg",
    "FlatGeobuf": "fields_inner.fgb",
    "ESRI Shapefile": "fields_inner.shp.zip",
}
for format_name, file_name in outputs.items():
    path = OUT / file_name
    inner.download(path, format=format_name)
    gdf = gpd.read_parquet(path) if format_name == "GeoParquet" else gpd.read_file(path)
    print(
        f"{format_name:15} {path.stat().st_size:7} bytes   EPSG:{gdf.crs.to_epsg()}   {len(gdf)} fields"
    )

## Interpretation

- Field A has a high NDVI (above 0.7) from the end of June to August, and its peak is
  in August. Fields B, C and D have a low NDVI in June and July, and their peak is
  later, in August or September. The
  [rasters to polygons notebook](https://sentinel-hub.github.io/titiler-openeo/notebooks/vector_raster_bridges/) shows that fields C and
  D are maize in 2024: maize grows late in the season.
- The relative NDVI shows the date of the peak of each field, independently of the
  level of the peak.
- A date without a value had clouds over the whole field: the SCL mask removes these
  pixels before the mean.
- One low value alone in a series (for example a value near 0.2 between two values
  above 0.5) is probably a cloud or a shadow that the SCL mask did not find.

## Limitations

- The requests are synchronous: keep the area and the season small.
- The default grid has 1024 pixels over the extent (see [Concepts](https://sentinel-hub.github.io/titiler-openeo/concepts/)).
- The SCL cloud mask is simple: it misses some clouds and cloud shadows.
- `merge_cubes` compares geometries exactly: a geometry that was reprojected and back
  is not equal to the original.
- The other known limits are in [Vector Data Cubes](https://sentinel-hub.github.io/titiler-openeo/vector-cube/#known-limits).

## References

- [Vector Data Cubes](https://sentinel-hub.github.io/titiler-openeo/vector-cube/) and
  [ADR 0014](https://github.com/sentinel-hub/titiler-openeo/blob/main/docs/adr/0014-vector-cubes-and-formats.md).
- openEO processes: [aggregate_spatial](https://processes.openeo.org/#aggregate_spatial),
  [merge_cubes](https://processes.openeo.org/#merge_cubes),
  [filter_bbox](https://processes.openeo.org/#filter_bbox).
- Contains modified Copernicus Sentinel data (2024), from the
  [Copernicus Data Space Ecosystem](https://dataspace.copernicus.eu/).